# Sprint 6 - Loss Comparison Runner

**Runner-only notebook.** Model, loss, sampler, training, evaluation, plotting, and reporting logic stays in the repository under `src/`, `scripts/`, and `configs/`.

Execution plan: `docs/exec-plans/active/006-sprint6-imbalance-loss-comparison.md`  
Runner boundary: `colab/README.md`

Before starting, confirm that the approved code revision is pushed and that Drive contains the Sprint 5 Graph A artifacts with the `S5F2_energy` edge table.

## Step 1 - Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 - Clone Or Use Repo Checkout

In [ ]:
%%bash
set -euo pipefail
pip install uv --quiet
REPO_URL="${REPO_URL:-https://github.com/YasinEkici/crispr-gnn-offtarget.git}"
REPO_DIR="${REPO_DIR:-/content/crispr-gnn-offtarget}"
GIT_REF="${GIT_REF:-main}"
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone "$REPO_URL" "$REPO_DIR"
fi
cd "$REPO_DIR"
git fetch --all --tags
git checkout "$GIT_REF"
echo "=== Commit SHA ==="
git rev-parse HEAD

## Step 3 - Dependency Sync And Runtime Check

In [ ]:
%%bash
set -euo pipefail
REPO_DIR="${REPO_DIR:-/content/crispr-gnn-offtarget}"
cd "$REPO_DIR"
uv sync
uv run python - <<'PY'
import torch
try:
    import torch_geometric
    pyg_version = torch_geometric.__version__
except Exception as exc:
    pyg_version = f'unavailable: {exc}'
print('torch         :', torch.__version__)
print('pyg           :', pyg_version)
print('cuda_available:', torch.cuda.is_available())
print('cuda_version  :', torch.version.cuda)
print('device        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
PY

## Step 4 - Copy Sprint 5 Graph A Artifacts From Drive

This copies existing Sprint 5 graph artifacts to the local path consumed by the Sprint 6 runner. It does not rebuild graph artifacts.

In [ ]:
%%bash
set -euo pipefail
REPO_DIR="${REPO_DIR:-/content/crispr-gnn-offtarget}"
cd "$REPO_DIR"
DRIVE_ROOTS=("/content/drive/MyDrive/crispr_gnn_offtarget" "/content/drive/MyDrive/crispr-gnn-offtarget")
GRAPH_SOURCE=""
for DRIVE_ROOT in "${DRIVE_ROOTS[@]}"; do
  CANDIDATE="$DRIVE_ROOT/data/processed/graphs/sprint5"
  if [ -d "$CANDIDATE/graph_a_minimal_physical_target" ]; then
    GRAPH_SOURCE="$CANDIDATE"
    break
  fi
done
if [ -z "$GRAPH_SOURCE" ]; then
  echo "Sprint 5 Graph A artifacts not found in Drive under data/processed/graphs/sprint5" >&2
  exit 1
fi
mkdir -p data/processed/graphs/sprint5
rsync -a "$GRAPH_SOURCE/" data/processed/graphs/sprint5/
echo "Copied Sprint 5 graph artifacts from: $GRAPH_SOURCE"
find data/processed/graphs/sprint5/graph_a_minimal_physical_target -maxdepth 2 -type f | sort | head -80

## Step 5 - Validate Graph A S5F2 Artifacts

In [ ]:
%%bash
set -euo pipefail
REPO_DIR="${REPO_DIR:-/content/crispr-gnn-offtarget}"
cd "$REPO_DIR"
PYTHONPATH=src uv run python - <<'PY'
from pathlib import Path
from crispr_gnn.graph.graph_schemas import GRAPH_A
from crispr_gnn.graph.pyg_dataset import Sprint3HeteroDataLoader

materialized = Sprint3HeteroDataLoader(Path('data/processed/graphs/sprint5')).load(GRAPH_A)
manifest = materialized.manifest
feature_tables = manifest.get('feature_tables', {})
if 'S5F2_energy' not in feature_tables and 's5f2_energy' not in feature_tables:
    raise SystemExit('Missing Sprint 5 S5F2_energy feature table')
print('graph_name:', manifest.get('graph_name'))
print('split_id:', manifest.get('split_id'))
print('label_scheme:', manifest.get('label_scheme'))
print('feature_tables:', feature_tables)
PY

## Step 6 - Run Sprint 6 Loss Comparison

In [ ]:
%%bash
set -euo pipefail
REPO_DIR="${REPO_DIR:-/content/crispr-gnn-offtarget}"
cd "$REPO_DIR"
RUN_ID="sprint6_loss_comparison_seed42_$(date -u +%Y%m%d_%H%M%S)"
uv run python scripts/run_sprint6_loss_comparison.py \
  --config configs/sweeps/sprint6_loss_comparison.yaml \
  --run-id "$RUN_ID"
echo "$RUN_ID" > /content/sprint6_loss_comparison_run_id.txt

## Step 7 - Copy Outputs Back To Drive

In [ ]:
%%bash
set -euo pipefail
REPO_DIR="${REPO_DIR:-/content/crispr-gnn-offtarget}"
cd "$REPO_DIR"
DRIVE_ROOTS=("/content/drive/MyDrive/crispr_gnn_offtarget" "/content/drive/MyDrive/crispr-gnn-offtarget")
DRIVE_ROOT=""
for CANDIDATE in "${DRIVE_ROOTS[@]}"; do
  if [ -d "$CANDIDATE" ]; then
    DRIVE_ROOT="$CANDIDATE"
    break
  fi
done
if [ -z "$DRIVE_ROOT" ]; then
  echo "No Drive project root found under MyDrive" >&2
  exit 1
fi
RUN_BASENAME=$(cat /content/sprint6_loss_comparison_run_id.txt)
LOCAL_OUT="outputs/sprint6/loss_comparison"
DRIVE_OUT="$DRIVE_ROOT/returned_outputs/$RUN_BASENAME"
mkdir -p "$DRIVE_ROOT/returned_outputs"
if [ -e "$DRIVE_OUT" ]; then
  echo "Output already exists in Drive: $DRIVE_OUT" >&2
  exit 1
fi
cp -r "$LOCAL_OUT" "$DRIVE_OUT"
echo "Copied to: $DRIVE_OUT"
find "$DRIVE_OUT" -maxdepth 4 -type f | sort | head -120

## Step 8 - Returned Artifact Checks

In [ ]:
%%bash
set -euo pipefail
REPO_DIR="${REPO_DIR:-/content/crispr-gnn-offtarget}"
cd "$REPO_DIR"
RUN_BASENAME=$(cat /content/sprint6_loss_comparison_run_id.txt)
OUT="outputs/sprint6/loss_comparison"
test -f "$OUT/sprint6_loss_comparison_results.csv"
test -f "$OUT/sprint6_loss_comparison_report.md"
test -f "$OUT/sprint6_loss_comparison_run_manifest.json"
test -f "$OUT/graph_artifact_provenance.json"
test -d "$OUT/diagnostics_sprint6"
test -d "$OUT/figures_sprint6"
test -d "$OUT/runs"
python - <<'PY'
import json
from pathlib import Path
manifest = json.loads(Path('outputs/sprint6/loss_comparison/sprint6_loss_comparison_run_manifest.json').read_text())
print('batch_id:', manifest['batch_id'])
print('headline_run_ids:', manifest['headline_run_ids'])
print('optional_runs_executed:', manifest['optional_runs_executed'])
print('run_count:', len(manifest['runs']))
if manifest['optional_runs_executed']:
    raise SystemExit('Unexpected optional run execution in headline notebook')
PY
find "$OUT" -maxdepth 3 -type f | sort | head -160
echo "Checked returned artifacts for $RUN_BASENAME"